# Autovalores e Autovetores

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import scipy.linalg as la

## Definição

Seja $A$ uma matriz quadrada. Um vetor não nulo $\mathbf{v}$ é um [autovetor](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors) para $A$ com [autovalor](https://en.wikipedia.org/wiki/Eigenvalues_and_eigenvectors) $\lambda$ se

$$
A\mathbf{v} = \lambda \mathbf{v}
$$

Rearranjando a equação, vemos que $\mathbf{v}$ é uma solução do sistema homogêneo de equações

$$
\left( A - \lambda I \right) \mathbf{v} = \mathbf{0}
$$

onde $I$ é a matriz identidade de tamanho $n$. Soluções não triviais existem apenas se a matriz $A - \lambda I$ for singular, o que significa que $\mathrm{det}(A - \lambda I) = 0$. Portanto, os autovalores de $A$ são as raízes do [polinômio característico](https://en.wikipedia.org/wiki/Characteristic_polynomial)

$$
p(\lambda) = \mathrm{det}(A - \lambda I)
$$

## scipy.linalg.eig

A função [`scipy.linalg.eig`](https://docs.scipy.org/doc/scipy/reference/generated/scipy.linalg.eig.html) calcula os autovalores e autovetores de uma matriz quadrada $A$.

Vamos considerar um exemplo simples com uma matriz diagonal:

In [ ]:
A = np.array([[1,0],[0,-2]])
print(A)

A função `la.eig` retorna uma tupla `(eigvals,eigvecs)` onde `eigvals` é um array NumPy 1D de números complexos que fornece os autovalores de $A$, e `eigvecs` é um array NumPy 2D com os autovetores correspondentes nas colunas:

In [ ]:
results = la.eig(A)

Os autovalores de $A$ são:

In [ ]:
print(results[0])

Os autovetores correspondentes são:

In [ ]:
print(results[1])

Podemos [desempacotar a tupla](../../python/sequences/#unpacking-a-sequence):

In [ ]:
eigvals, eigvecs = la.eig(A)
print(eigvals)

In [ ]:
print(eigvecs)

Se soubermos que os autovalores são números reais (ou seja, se $A$ for simétrica), podemos usar o método `.real` do array NumPy para converter o array de autovalores para números reais:

In [ ]:
eigvals = eigvals.real
print(eigvals)

Note que a posição de um autovalor no array `eigvals` corresponde à coluna em `eigvecs` com seu autovetor:

In [ ]:
lambda1 = eigvals[1]
print(lambda1)

In [ ]:
v1 = eigvecs[:,1].reshape(2,1)
print(v1)

In [ ]:
A @ v1

In [ ]:
lambda1 * v1

## Exemplos

### Matrizes Simétricas

Os autovalores de uma [matriz simétrica](https://en.wikipedia.org/wiki/Symmetric_matrix) são sempre reais e os autovetores são sempre ortogonais! Vamos verificar esses fatos com algumas matrizes aleatórias:

In [ ]:
n = 4
P = np.random.randint(0,10,(n,n))
print(P)

Crie a matriz simétrica $S = P P^T$:

In [ ]:
S = P @ P.T
print(S)

Vamos desempacotar os autovalores e autovetores de $S$:

In [ ]:
evals, evecs = la.eig(S)
print(evals)

Os autovalores têm todos parte imaginária nula e, portanto, são de fato números reais:

In [ ]:
evals = evals.real
print(evals)

Os autovetores correspondentes de $A$ são:

In [ ]:
print(evecs)

Vamos verificar se os autovetores são ortogonais entre si:

In [ ]:
v1 = evecs[:,0] # A primeira coluna é o primeiro autovetor
print(v1)

In [ ]:
v2 = evecs[:,1] # A segunda coluna é o segundo autovetor
print(v2)

In [ ]:
v1 @ v2

O produto escalar dos autovetores $\mathbf{v}_1$ e $\mathbf{v}_2$ é zero (o número acima é *muito* próximo de zero e se deve a erros de arredondamento nos cálculos) e, portanto, eles são ortogonais!

### Diagonalização

Uma matriz quadrada $M$ é [diagonalizável](https://en.wikipedia.org/wiki/Diagonalizable_matrix) se for semelhante a uma matriz diagonal. Em outras palavras, $M$ é diagonalizável se existir uma matriz invertível $P$ tal que $D = P^{-1}MP$ seja uma matriz diagonal.

Um belo resultado em álgebra linear é que uma matriz quadrada $M$ de tamanho $n$ é diagonalizável se, e somente se, $M$ tiver $n$ autovetores independentes. Além disso, $M = PDP^{-1}$ onde as colunas de $P$ são os autovetores de $M$ e $D$ tem os autovalores correspondentes ao longo da diagonal.

Vamos usar isso para construir uma matriz com autovalores dados $\lambda_1 = 3, \lambda_2 = 1$, e autovetores $v_1 = [1,1]^T, v_2 = [1,-1]^T$.

In [ ]:
P = np.array([[1,1],[1,-1]])
print(P)

In [ ]:
D = np.diag((3,1))
print(D)

In [ ]:
M = P @ D @ la.inv(P)
print(M)

Vamos verificar se os autovalores de $M$ são 3 e 1:

In [ ]:
evals, evecs = la.eig(M)
print(evals)

Verifique os autovetores:

In [ ]:
print(evecs)

### Potências de Matrizes

Seja $M$ uma matriz quadrada. Calcular potências de $M$ por multiplicação de matrizes

$$
M^k = \underbrace{M M \cdots M}_k
$$

é computacionalmente caro. Em vez disso, vamos usar a diagonalização para calcular $M^k$ de forma mais eficiente

$$
M^k = \left( P D P^{-1} \right)^k = \underbrace{P D P^{-1} P D P^{-1} \cdots P D P^{-1}}_k = P D^k P^{-1}
$$

Vamos calcular $M^{20}$ de ambas as formas e comparar o tempo de execução.

In [ ]:
Pinv = la.inv(P)

In [ ]:
k = 20

In [ ]:
%%timeit
result = M.copy()
for _ in range(1,k):
    result = result @ M

Vamos usar a diagonalização para fazer o mesmo cálculo.

In [ ]:
%%timeit
P @ D**k @ Pinv

A diagonalização calcula $M^{k}$ muito mais rápido!